In [1]:
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import pandas as pd
import numpy as np
import joblib

tqdm.pandas()

In [7]:
df = pd.read_csv('../all_datasets_transformers.csv')
df=df.dropna(how='any')
df.head()

,text,type,label
0,"USER Because while other groups can evolve, th...",cyberbullying,1
1,New Discussion While I understand your poin...,not_cyberbullying,0
2,No page was reviewed by Tentinator This rec...,not_cyberbullying,0
3,Laughing big grin or laugh with glasses...,not_cyberbullying,0
4,USER you should not have taken the statue away...,cyberbullying,1


In [9]:
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.2, random_state=42,stratify=df['label'])

In [3]:
def load_glove_embeddings(file_path):
    glove_vectors = dict()
    with open(file_path, encoding='utf-8') as file:
        for line in file:
            values = line.split()
            word = values[0]
            vectors = np.asarray(values[1:], dtype=float)
            glove_vectors[word] = vectors
    return glove_vectors

In [5]:
glove_embeddings=load_glove_embeddings('./glove.twitter.27B.200d.txt')    

In [6]:
def get_glove_vectors(x, glove_vectors, vec_shape=200):
    array = np.zeros(vec_shape)
    text = str(x).split()
    
    count = 0

    for word in text:
        if word in glove_vectors:
            array += glove_vectors[word]  
            count += 1 

    if count > 0:
        return array / count
    else:
        return array

In [10]:
X_glove = np.array([get_glove_vectors(text, glove_embeddings) for text in df['text']])
y = df['label']

In [11]:
X_train_glove, X_test_glove, y_train, y_test = train_test_split(X_glove, y, test_size=0.2,stratify=y, random_state=42)

print(f"X_train_glove: {X_train_glove.shape}")
print(f"X_test_glove: {X_test_glove.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

X_train_glove: (120720, 200)
X_test_glove: (30181, 200)
y_train: (120720,)
y_test: (30181,)


In [12]:
joblib.dump(X_train_glove, 'X_train_glove.pkl')
joblib.dump(X_test_glove, 'X_test_glove.pkl')
joblib.dump(y_train, 'y_train_glove.pkl')
joblib.dump(y_test, 'y_test_glove.pkl')


['y_test_glove.pkl']